# Week 38, Day 2: Project: domain classes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w038d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('errors.py', 'w', encoding='utf-8') as f:
    f.write('"""Every error the library raises on purpose."""\n\n\nclass LibraryError(Exception):\n    """Base class for the library\'s own errors."""\n\n\nclass InvalidInput(LibraryError, ValueError):\n    """Something typed or loaded doesn\'t make sense."""\n\n\nclass NotFound(LibraryError, LookupError):\n    """There\'s no book or member with that id."""\n\n\nclass NotAllowed(LibraryError):\n    """The library\'s rules don\'t allow this right now."""\n\n\nclass StorageError(LibraryError):\n    """The library\'s data couldn\'t be read or saved."""\n')
with open('values.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and checking the values people type, and showing money."""\nfrom errors import InvalidInput\n\nKINDS = {"B": "book", "M": "member"}\n\n\ndef clean_isbn(text):\n    """\'978-0-14-118776-1\' as \'9780141187761\'. Raises InvalidInput."""\n    shown = text.strip()\n    digits = shown.replace("-", "").replace(" ", "")\n    if len(digits) != 13 or not (digits.isascii() and digits.isdigit()):\n        raise InvalidInput(f"an ISBN has 13 digits: {shown!r}")\n    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(digits))\n    if total % 10 != 0:\n        raise InvalidInput(f"wrong check digit: {shown!r}")\n    return digits\n\n\ndef clean_text(text, what, longest):\n    """\'  Things   Fall Apart \' as \'Things Fall Apart\'. Raises InvalidInput."""\n    cleaned = " ".join(text.split())\n    if not cleaned:\n        raise InvalidInput(f"{what} is missing")\n    if len(cleaned) > longest:\n        raise InvalidInput(f"{what} is longer than {longest} characters")\n    return cleaned\n\n\ndef parse_id(text, kind):\n    """\' b12 \' as \'B12\', for kind \'B\'. Raises InvalidInput."""\n    cleaned = text.strip().upper()\n    number = cleaned[1:]\n    if not cleaned.startswith(kind) or not (number.isascii() and number.isdigit()) or number.startswith("0"):\n        raise InvalidInput(f"not a {KINDS[kind]} id: {text.strip()!r}")\n    return cleaned\n\n\ndef parse_pence(text):\n    """\'£1.20\' or \'0.6\' as whole pence. Raises InvalidInput."""\n    shown = text.strip()\n    pounds, dot, pence = shown.removeprefix("£").partition(".")\n    if (not (pounds.isascii() and pounds.isdigit())\n            or (dot and not (pence.isascii() and pence.isdigit() and len(pence) <= 2))):\n        raise InvalidInput(f"not an amount: {shown!r}")\n    return int(pounds) * 100 + (int(pence.ljust(2, "0")) if dot else 0)\n\n\ndef format_pence(pence):\n    """60 as \'£0.60\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef check_all(**checks):\n    """Run each check (a function of no arguments); return their results by name, or raise one\n    InvalidInput listing every problem, in order."""\n    results, problems = {}, []\n    for name, check in checks.items():\n        try:\n            results[name] = check()\n        except InvalidInput as err:\n            problems.append(str(err))\n    if problems:\n        raise InvalidInput("; ".join(problems))\n    return results\n')

### Exercise: Step 7: books and members

Write two dataclasses. `Book` has `book_id`, `isbn`, `title` and `author` (all text), then `borrower` (a member id, default `None`) and `holds` (a list of member ids waiting, default a **new** empty list for each book). It has a property `on_loan` (true when there's a borrower) and a method `describe()` returning `"B1: Things Fall Apart by Chinua Achebe"`. `Member` has `member_id`, `name`, `max_loans` (default 3) and `owes` (pence, default 0).

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

from errors import InvalidInput, NotAllowed
from values import format_pence

LOAN_DAYS = 21
FINE_PER_DAY = 20      # pence for each day late
MAX_FINE = 500         # the most one loan can cost
FINE_LIMIT = 500       # owing this much stops a member borrowing

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\n\nimport dataclasses\ndef test_book():\n    \"books\"\n    assert dataclasses.is_dataclass(Book) and dataclasses.is_dataclass(Member)\n    b = Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\")\n    assert b.borrower is None and b.holds == [] and b.on_loan is False\n    assert b.describe() == \"B1: Things Fall Apart by Chinua Achebe\"\n    b.borrower = \"M1\"\n    assert b.on_loan is True\n    c = Book(\"B2\", \"9780435950828\", \"Arrow of God\", \"Chinua Achebe\")\n    c.holds.append(\"M2\")\n    assert b.holds == [], \"each book needs its own list of holds\"\n    assert [f.name for f in dataclasses.fields(Book)] == [\"book_id\", \"isbn\", \"title\", \"author\", \"borrower\", \"holds\"]\ndef test_member():\n    \"members\"\n    m = Member(\"M1\", \"Ada Obi\")\n    assert (m.max_loans, m.owes) == (3, 0) and Member(\"M2\", \"Bo\", 5, 40).owes == 40")

### Exercise: Step 8: loans and fines

Write the dataclass `Loan` with `loan_id` (an int), `book_id`, `member_id`, `out` and `due` (dates) and `returned` (a date, default `None`). Give it:

- a property `is_open`, true until the book is returned;
- `days_late(today)`: how many days after `due` the book came back (or, if it's still out, how many days after `due` `today` is), never below 0;
- `fine(today)`: `FINE_PER_DAY` for each day late, but no more than `MAX_FINE`.

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

from errors import InvalidInput, NotAllowed
from values import format_pence

LOAN_DAYS = 21
FINE_PER_DAY = 20      # pence for each day late
MAX_FINE = 500         # the most one loan can cost
FINE_LIMIT = 500       # owing this much stops a member borrowing

@dataclass
class Book:
    book_id: str
    isbn: str
    title: str
    author: str
    borrower: str | None = None
    holds: list[str] = field(default_factory=list)

    @property
    def on_loan(self):
        return self.borrower is not None

    def describe(self):
        return f"{self.book_id}: {self.title} by {self.author}"


@dataclass
class Member:
    member_id: str
    name: str
    max_loans: int = 3
    owes: int = 0


___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\n\ndef test_open():\n    \"open loans\"\n    loan = Loan(1, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23))\n    assert loan.is_open and loan.returned is None\n    assert loan.days_late(D(2026, 11, 20)) == 0 and loan.days_late(D(2026, 11, 23)) == 0 and loan.days_late(D(2026, 11, 26)) == 3\n    assert loan.fine(D(2026, 11, 26)) == 60 and loan.fine(D(2026, 11, 1)) == 0\n    assert loan.fine(D(2027, 1, 1)) == 500, \"fines stop at MAX_FINE\"\ndef test_returned():\n    \"returned loans\"\n    loan = Loan(2, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23), D(2026, 11, 25))\n    assert not loan.is_open\n    assert loan.days_late(D(2027, 3, 1)) == 2, \"once returned, the return date counts, not today\"\n    assert loan.fine(D(2027, 3, 1)) == 40")

### Exercise: Step 9: the borrowing rules

Write `check_can_borrow(member, book, open_loans, today)`, where `open_loans` is the member's open loans. It returns nothing if the member may borrow the book, and otherwise raises `NotAllowed` with the **first** of these reasons that applies:

1. `"B1 is on loan"`;
2. `"B1 is being kept for someone else"`, if the book has holds and the first isn't this member;
3. `"Ada Obi already has 3 books"` (the number they have), if they're at their `max_loans`;
4. `"Ada Obi has an overdue book"`, if any open loan is late today;
5. `"Ada Obi owes £5.00"` (with `format_pence`), if they owe `FINE_LIMIT` or more.

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

from errors import InvalidInput, NotAllowed
from values import format_pence

LOAN_DAYS = 21
FINE_PER_DAY = 20      # pence for each day late
MAX_FINE = 500         # the most one loan can cost
FINE_LIMIT = 500       # owing this much stops a member borrowing

@dataclass
class Book:
    book_id: str
    isbn: str
    title: str
    author: str
    borrower: str | None = None
    holds: list[str] = field(default_factory=list)

    @property
    def on_loan(self):
        return self.borrower is not None

    def describe(self):
        return f"{self.book_id}: {self.title} by {self.author}"


@dataclass
class Member:
    member_id: str
    name: str
    max_loans: int = 3
    owes: int = 0


@dataclass
class Loan:
    loan_id: int
    book_id: str
    member_id: str
    out: date
    due: date
    returned: date | None = None

    @property
    def is_open(self):
        return self.returned is None

    def days_late(self, today):
        end = self.returned or today
        return max(0, (end - self.due).days)

    def fine(self, today):
        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)


___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\n\nTODAY = D(2026, 11, 26)\ndef loan(n, due):\n    return Loan(n, f\"B{n + 10}\", \"M1\", D(2026, 11, 1), due)\ndef reason(member, book, loans):\n    try:\n        check_can_borrow(member, book, loans, TODAY)\n    except NotAllowed as err:\n        return str(err)\n    return None\ndef test_allowed():\n    \"allowed\"\n    ada = Member(\"M1\", \"Ada Obi\")\n    assert reason(ada, Book(\"B1\", \"x\", \"T\", \"A\"), []) is None\n    assert reason(ada, Book(\"B1\", \"x\", \"T\", \"A\", holds=[\"M1\", \"M2\"]), [loan(1, TODAY)]) is None, \"the first in the queue may borrow; due today isn't late\"\n    assert reason(Member(\"M1\", \"Ada Obi\", owes=499), Book(\"B1\", \"x\", \"T\", \"A\"), []) is None\ndef test_refused():\n    \"each reason\"\n    ada = Member(\"M1\", \"Ada Obi\")\n    assert reason(ada, Book(\"B1\", \"x\", \"T\", \"A\", borrower=\"M2\"), []) == \"B1 is on loan\"\n    assert reason(ada, Book(\"B1\", \"x\", \"T\", \"A\", holds=[\"M2\", \"M1\"]), []) == \"B1 is being kept for someone else\"\n    three = [loan(1, D(2026, 12, 1)), loan(2, D(2026, 12, 1)), loan(3, D(2026, 12, 1))]\n    assert reason(ada, Book(\"B1\", \"x\", \"T\", \"A\"), three) == \"Ada Obi already has 3 books\"\n    assert reason(Member(\"M1\", \"Ada Obi\", max_loans=5), Book(\"B1\", \"x\", \"T\", \"A\"), three) is None\n    assert reason(ada, Book(\"B1\", \"x\", \"T\", \"A\"), [loan(1, D(2026, 11, 25))]) == \"Ada Obi has an overdue book\"\n    assert reason(Member(\"M1\", \"Ada Obi\", owes=500), Book(\"B1\", \"x\", \"T\", \"A\"), []) == \"Ada Obi owes \u00a35.00\"\ndef test_order():\n    \"the first reason\"\n    m = Member(\"M1\", \"Ada Obi\", max_loans=1, owes=900)\n    assert reason(m, Book(\"B1\", \"x\", \"T\", \"A\", borrower=\"M2\", holds=[\"M3\"]), [loan(1, D(2026, 11, 1))]) == \"B1 is on loan\"\n    assert reason(m, Book(\"B1\", \"x\", \"T\", \"A\"), [loan(1, D(2026, 11, 1))]) == \"Ada Obi already has 1 books\"")

### Exercise: Step 10: lending and returning

Write two functions:

- `lend(book, member, open_loans, today, loan_id)`: check the rules with `check_can_borrow` (changing nothing if it refuses); if the book had holds, remove the first (it must be this member's); set the book's borrower; and return a new `Loan` that's due `LOAN_DAYS` days from today.
- `give_back(book, loan, member, today)`: close the loan (set `returned`), clear the book's borrower, add the loan's fine to what the member owes, and return the fine. The service will always pass a matching open loan, so check that with an `assert` (a mismatch would be a bug in the program, not a mistake by a person).

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

from errors import InvalidInput, NotAllowed
from values import format_pence

LOAN_DAYS = 21
FINE_PER_DAY = 20      # pence for each day late
MAX_FINE = 500         # the most one loan can cost
FINE_LIMIT = 500       # owing this much stops a member borrowing

@dataclass
class Book:
    book_id: str
    isbn: str
    title: str
    author: str
    borrower: str | None = None
    holds: list[str] = field(default_factory=list)

    @property
    def on_loan(self):
        return self.borrower is not None

    def describe(self):
        return f"{self.book_id}: {self.title} by {self.author}"


@dataclass
class Member:
    member_id: str
    name: str
    max_loans: int = 3
    owes: int = 0


@dataclass
class Loan:
    loan_id: int
    book_id: str
    member_id: str
    out: date
    due: date
    returned: date | None = None

    @property
    def is_open(self):
        return self.returned is None

    def days_late(self, today):
        end = self.returned or today
        return max(0, (end - self.due).days)

    def fine(self, today):
        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)


def check_can_borrow(member, book, open_loans, today):
    """Raise NotAllowed with the first reason the member can't borrow the book today."""
    if book.on_loan:
        raise NotAllowed(f"{book.book_id} is on loan")
    if book.holds and book.holds[0] != member.member_id:
        raise NotAllowed(f"{book.book_id} is being kept for someone else")
    if len(open_loans) >= member.max_loans:
        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")
    if any(loan.days_late(today) for loan in open_loans):
        raise NotAllowed(f"{member.name} has an overdue book")
    if member.owes >= FINE_LIMIT:
        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")


___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\n\ndef test_lend():\n    \"lending\"\n    book, ada = Book(\"B1\", \"x\", \"T\", \"A\"), Member(\"M1\", \"Ada Obi\")\n    loan = lend(book, ada, [], D(2026, 11, 2), 7)\n    assert loan == Loan(7, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23)) and book.borrower == \"M1\"\ndef test_holds():\n    \"the queue\"\n    book = Book(\"B1\", \"x\", \"T\", \"A\", holds=[\"M2\", \"M3\"])\n    try:\n        lend(book, Member(\"M3\", \"Chi\"), [], D(2026, 11, 2), 1)\n        assert False, \"M2 is first in the queue\"\n    except NotAllowed:\n        assert book.holds == [\"M2\", \"M3\"] and book.borrower is None, \"a refusal changes nothing\"\n    lend(book, Member(\"M2\", \"Bo\"), [], D(2026, 11, 2), 2)\n    assert book.holds == [\"M3\"] and book.borrower == \"M2\"\ndef test_give_back():\n    \"returning\"\n    book, ada = Book(\"B1\", \"x\", \"T\", \"A\"), Member(\"M1\", \"Ada Obi\", owes=20)\n    loan = lend(book, ada, [], D(2026, 11, 2), 1)\n    assert give_back(book, loan, ada, D(2026, 11, 26)) == 60\n    assert loan.returned == D(2026, 11, 26) and not loan.is_open and book.borrower is None and ada.owes == 80\n    other = lend(book, ada, [], D(2026, 12, 1), 2)\n    assert give_back(book, other, ada, D(2026, 12, 3)) == 0 and ada.owes == 80\n    try:\n        give_back(book, other, ada, D(2026, 12, 4))\n        assert False, \"a closed loan is a bug: assert\"\n    except AssertionError:\n        pass")

### Exercise: Step 11: reservations

Write `reserve(book, member)`, which adds the member to the end of the book's queue and returns their place in it (1 for the first). It raises `NotAllowed`:

- `"B1 is on the shelf: borrow it instead"` if the book isn't on loan and nobody is waiting;
- `"Ada Obi already has B1"` if the member is the borrower;
- `"Ada Obi is already waiting for B1"` if they're already in the queue.

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

from errors import InvalidInput, NotAllowed
from values import format_pence

LOAN_DAYS = 21
FINE_PER_DAY = 20      # pence for each day late
MAX_FINE = 500         # the most one loan can cost
FINE_LIMIT = 500       # owing this much stops a member borrowing

@dataclass
class Book:
    book_id: str
    isbn: str
    title: str
    author: str
    borrower: str | None = None
    holds: list[str] = field(default_factory=list)

    @property
    def on_loan(self):
        return self.borrower is not None

    def describe(self):
        return f"{self.book_id}: {self.title} by {self.author}"


@dataclass
class Member:
    member_id: str
    name: str
    max_loans: int = 3
    owes: int = 0


@dataclass
class Loan:
    loan_id: int
    book_id: str
    member_id: str
    out: date
    due: date
    returned: date | None = None

    @property
    def is_open(self):
        return self.returned is None

    def days_late(self, today):
        end = self.returned or today
        return max(0, (end - self.due).days)

    def fine(self, today):
        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)


def check_can_borrow(member, book, open_loans, today):
    """Raise NotAllowed with the first reason the member can't borrow the book today."""
    if book.on_loan:
        raise NotAllowed(f"{book.book_id} is on loan")
    if book.holds and book.holds[0] != member.member_id:
        raise NotAllowed(f"{book.book_id} is being kept for someone else")
    if len(open_loans) >= member.max_loans:
        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")
    if any(loan.days_late(today) for loan in open_loans):
        raise NotAllowed(f"{member.name} has an overdue book")
    if member.owes >= FINE_LIMIT:
        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")


def lend(book, member, open_loans, today, loan_id):
    """Lend the book, or raise NotAllowed. Returns the new Loan."""
    check_can_borrow(member, book, open_loans, today)
    if book.holds:
        book.holds.pop(0)
    book.borrower = member.member_id
    return Loan(loan_id, book.book_id, member.member_id, today, today + timedelta(days=LOAN_DAYS))


def give_back(book, loan, member, today):
    """Close the loan and add any fine to what the member owes. Returns the fine."""
    assert loan.is_open and loan.book_id == book.book_id and loan.member_id == member.member_id
    loan.returned = today
    book.borrower = None
    fine = loan.fine(today)
    member.owes += fine
    return fine


___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\n\ndef test_reserve():\n    \"reserving\"\n    book = Book(\"B1\", \"x\", \"T\", \"A\", borrower=\"M1\")\n    assert reserve(book, Member(\"M2\", \"Bo Lee\")) == 1 and reserve(book, Member(\"M3\", \"Chi\")) == 2 and book.holds == [\"M2\", \"M3\"]\n    for member, message in [(Member(\"M1\", \"Ada Obi\"), \"Ada Obi already has B1\"), (Member(\"M2\", \"Bo Lee\"), \"Bo Lee is already waiting for B1\")]:\n        try:\n            reserve(book, member)\n            assert False, message\n        except NotAllowed as err:\n            assert str(err) == message, f\"got {str(err)!r}\"\n    assert book.holds == [\"M2\", \"M3\"]\n    kept = Book(\"B2\", \"x\", \"T\", \"A\", holds=[\"M2\"])\n    assert reserve(kept, Member(\"M4\", \"Dee\")) == 2, \"a book kept for someone can still be reserved\"\n    try:\n        reserve(Book(\"B3\", \"x\", \"T\", \"A\"), Member(\"M1\", \"Ada Obi\"))\n        assert False\n    except NotAllowed as err:\n        assert str(err) == \"B3 is on the shelf: borrow it instead\"")

### Exercise: Step 12: paying fines

Write `pay(member, pence)`, which takes a payment off what the member owes and returns what's left. A payment of 0 or less raises `InvalidInput("a payment must be more than £0")`; more than they owe raises `NotAllowed("Ada Obi only owes £0.60")`.

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

from errors import InvalidInput, NotAllowed
from values import format_pence

LOAN_DAYS = 21
FINE_PER_DAY = 20      # pence for each day late
MAX_FINE = 500         # the most one loan can cost
FINE_LIMIT = 500       # owing this much stops a member borrowing

@dataclass
class Book:
    book_id: str
    isbn: str
    title: str
    author: str
    borrower: str | None = None
    holds: list[str] = field(default_factory=list)

    @property
    def on_loan(self):
        return self.borrower is not None

    def describe(self):
        return f"{self.book_id}: {self.title} by {self.author}"


@dataclass
class Member:
    member_id: str
    name: str
    max_loans: int = 3
    owes: int = 0


@dataclass
class Loan:
    loan_id: int
    book_id: str
    member_id: str
    out: date
    due: date
    returned: date | None = None

    @property
    def is_open(self):
        return self.returned is None

    def days_late(self, today):
        end = self.returned or today
        return max(0, (end - self.due).days)

    def fine(self, today):
        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)


def check_can_borrow(member, book, open_loans, today):
    """Raise NotAllowed with the first reason the member can't borrow the book today."""
    if book.on_loan:
        raise NotAllowed(f"{book.book_id} is on loan")
    if book.holds and book.holds[0] != member.member_id:
        raise NotAllowed(f"{book.book_id} is being kept for someone else")
    if len(open_loans) >= member.max_loans:
        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")
    if any(loan.days_late(today) for loan in open_loans):
        raise NotAllowed(f"{member.name} has an overdue book")
    if member.owes >= FINE_LIMIT:
        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")


def lend(book, member, open_loans, today, loan_id):
    """Lend the book, or raise NotAllowed. Returns the new Loan."""
    check_can_borrow(member, book, open_loans, today)
    if book.holds:
        book.holds.pop(0)
    book.borrower = member.member_id
    return Loan(loan_id, book.book_id, member.member_id, today, today + timedelta(days=LOAN_DAYS))


def give_back(book, loan, member, today):
    """Close the loan and add any fine to what the member owes. Returns the fine."""
    assert loan.is_open and loan.book_id == book.book_id and loan.member_id == member.member_id
    loan.returned = today
    book.borrower = None
    fine = loan.fine(today)
    member.owes += fine
    return fine


def reserve(book, member):
    """Join the queue for a book. Returns the member's place in it."""
    if not book.on_loan and not book.holds:
        raise NotAllowed(f"{book.book_id} is on the shelf: borrow it instead")
    if book.borrower == member.member_id:
        raise NotAllowed(f"{member.name} already has {book.book_id}")
    if member.member_id in book.holds:
        raise NotAllowed(f"{member.name} is already waiting for {book.book_id}")
    book.holds.append(member.member_id)
    return len(book.holds)


___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\n\ndef test_pay():\n    \"paying\"\n    ada = Member(\"M1\", \"Ada Obi\", owes=60)\n    assert pay(ada, 20) == 40 and ada.owes == 40 and pay(ada, 40) == 0\n    for amount, error, message in [(0, InvalidInput, \"a payment must be more than \u00a30\"), (5, NotAllowed, \"Ada Obi only owes \u00a30.00\")]:\n        try:\n            pay(ada, amount)\n            assert False\n        except error as err:\n            assert str(err) == message, f\"got {str(err)!r}\"")

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

from errors import InvalidInput, NotAllowed
from values import format_pence

LOAN_DAYS = 21
FINE_PER_DAY = 20      # pence for each day late
MAX_FINE = 500         # the most one loan can cost
FINE_LIMIT = 500       # owing this much stops a member borrowing


@dataclass
class Book:
    book_id: str
    isbn: str
    title: str
    author: str
    borrower: str | None = None
    holds: list[str] = field(default_factory=list)

    @property
    def on_loan(self):
        return self.borrower is not None

    def describe(self):
        return f"{self.book_id}: {self.title} by {self.author}"


@dataclass
class Member:
    member_id: str
    name: str
    max_loans: int = 3
    owes: int = 0


@dataclass
class Loan:
    loan_id: int
    book_id: str
    member_id: str
    out: date
    due: date
    returned: date | None = None

    @property
    def is_open(self):
        return self.returned is None

    def days_late(self, today):
        end = self.returned or today
        return max(0, (end - self.due).days)

    def fine(self, today):
        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)


def check_can_borrow(member, book, open_loans, today):
    """Raise NotAllowed with the first reason the member can't borrow the book today."""
    if book.on_loan:
        raise NotAllowed(f"{book.book_id} is on loan")
    if book.holds and book.holds[0] != member.member_id:
        raise NotAllowed(f"{book.book_id} is being kept for someone else")
    if len(open_loans) >= member.max_loans:
        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")
    if any(loan.days_late(today) for loan in open_loans):
        raise NotAllowed(f"{member.name} has an overdue book")
    if member.owes >= FINE_LIMIT:
        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")


def lend(book, member, open_loans, today, loan_id):
    """Lend the book, or raise NotAllowed. Returns the new Loan."""
    check_can_borrow(member, book, open_loans, today)
    if book.holds:
        book.holds.pop(0)
    book.borrower = member.member_id
    return Loan(loan_id, book.book_id, member.member_id, today, today + timedelta(days=LOAN_DAYS))


def give_back(book, loan, member, today):
    """Close the loan and add any fine to what the member owes. Returns the fine."""
    assert loan.is_open and loan.book_id == book.book_id and loan.member_id == member.member_id
    loan.returned = today
    book.borrower = None
    fine = loan.fine(today)
    member.owes += fine
    return fine


def reserve(book, member):
    """Join the queue for a book. Returns the member's place in it."""
    if not book.on_loan and not book.holds:
        raise NotAllowed(f"{book.book_id} is on the shelf: borrow it instead")
    if book.borrower == member.member_id:
        raise NotAllowed(f"{member.name} already has {book.book_id}")
    if member.member_id in book.holds:
        raise NotAllowed(f"{member.name} is already waiting for {book.book_id}")
    book.holds.append(member.member_id)
    return len(book.holds)


def pay(member, pence):
    """Take a payment towards the member's fines. Returns what they still owe."""
    if pence <= 0:
        raise InvalidInput("a payment must be more than £0")
    if pence > member.owes:
        raise NotAllowed(f"{member.name} only owes {format_pence(member.owes)}")
    member.owes -= pence
    return member.owes


today = date(2026, 11, 2)
book = Book("B1", "9780141187761", "Things Fall Apart", "Chinua Achebe")
ada, bo = Member("M1", "Ada Obi"), Member("M2", "Bo Lee")

loan = lend(book, ada, [], today, 1)
print("lent:", loan)
print("Bo is number", reserve(book, bo), "in the queue")

back = date(2026, 11, 26)
print("fine:", format_pence(give_back(book, loan, ada, back)), "| Ada owes", format_pence(ada.owes))
for member in (ada, bo):
    try:
        lend(book, member, [], back, 2)
        print(member.name, "borrowed it")
    except NotAllowed as err:
        print("refused:", err)
print("Ada still owes", format_pence(pay(ada, 30)))
print(book)

> **Rules as functions.** Why are `lend` and `reserve` functions, rather than methods on `Book` or `Member`? Because each rule involves **several** objects: lending needs the book, the member, the member's other loans and the date. Putting it on `Book` would make the book know about members' loans; putting it on `Member` would make members know about holds. A function that takes everything it needs keeps each class small and the rule in one readable place. (Some designs gather these into a `Library` class in the domain; that's a fine choice too. Week 31's "single responsibility" is the test: each piece has one reason to change.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

Yesterday's code is in the files below: `errors.py` with the exception family, and `values.py` with `clean_isbn`, `clean_text`, `parse_id`, `parse_pence`, `format_pence` and `check_all`. Today's code is `domain.py`. In each editor, it starts with the imports and the library's numbers, which every rule uses:

| Name | Value | Meaning |
|---|---|---|
| `LOAN_DAYS` | 21 | a loan lasts three weeks |
| `FINE_PER_DAY` | 20 | 20p for each day late |
| `MAX_FINE` | 500 | at most £5 for one loan |
| `FINE_LIMIT` | 500 | owing £5 or more stops a member borrowing |

Naming the numbers (Week 13) means a change of policy, say 30p a day, is one edit in one place, and the tests can use the names too.

**What the domain knows, and what it doesn't.** A `Book` knows who has it and who's waiting; a `Member` knows how many books they may have and what they owe; a `Loan` knows its dates, and so it can work out whether it's late and what it costs. The domain doesn't know how these are stored, how ids are chosen, or what today's date is: those are passed in. That's what keeps it simple to test: no files, no clock, just objects in and objects out.

## Step 7: books and members

Dataclasses (Week 32) write `__init__`, `__repr__` and `__eq__` for you. The one trap is a default that's a list: every book must have its **own** list of holds, which is what `field(default_factory=list)` gives.

## Step 8: loans and fines

A loan is a record of one book going out and coming back. It can answer questions about itself: is it still open? how late is it? what does it cost? Each answer depends on today's date, which is passed in.

## Step 9: the borrowing rules

The rules about who may borrow go in one function that checks them in a fixed order and raises at the first that fails. One function, rather than checks scattered through the program, means there's one place to read the policy and one place to change it.

## Step 10: lending and returning

Now the two changes. `lend` uses Week 37's pattern: **check everything, then change**. Because `check_can_borrow` runs first, a refused loan leaves the book exactly as it was.

`give_back` is different. A person can't cause a mismatched loan: the service looks up the loan for the book, so if they don't match, the program has a bug. That's a job for `assert`, not `raise` (Week 37 Day 5).

## Step 11: reservations

A popular book has a queue. Members join it while the book is out; when it comes back, `lend` already lets only the first in the queue borrow it. A book that's on the shelf with nobody waiting can't be reserved: the member should just borrow it.

## Step 12: paying fines

Fines collect in `Member.owes`. A payment can't be zero or negative (that's bad input, `InvalidInput`), and can't be more than is owed (the rules say no, `NotAllowed`).

## Checkpoint

Here's the finished `domain.py` telling a short story: Ada borrows a book, Bo reserves it, Ada brings it back three days late, and the book is kept for Bo.

Notice the last line: the book's `holds` list is empty again, because Bo's place in the queue became Bo's loan.

## Quiz

## Before you go

Put today's code in `domain.py` in your `shelf` folder, run the checkpoint story as a quick check, and commit. Tomorrow: the persistence layer, keeping books, members and loans in memory and in a JSON file that can't be corrupted by a crash.